# TDCOSMO H0 vs lens redshift exploratory analysis

This notebook checks whether the TDCOSMO lens sample shows a trend in `H0` with lens redshift `z_d`.

We compare a linear model, `H0 = a + b*z`, against a constant model, `H0 = c`.

This is an exploratory test, not a final cosmological result.


In [ ]:
import numpy as np
import emcee
import matplotlib.pyplot as plt

# Lens redshift data
z = np.array([
    0.230,
    0.295,
    0.311,
    0.407,
    0.454,
    0.590,
    0.597,
    0.630,
    0.657,
    0.729,
    0.745,
])

# Measured H0 values in km/s/Mpc
H0 = np.array([
    65.0,
    78.2,
    81.1,
    73.2,
    71.7,
    75.5,
    74.2,
    71.0,
    71.6,
    64.2,
    68.9,
])

# Asymmetric lower and upper error bars
sigma_minus = np.array([
    14.0,
    3.4,
    7.1,
    4.7,
    4.5,
    5.8,
    3.0,
    3.3,
    4.9,
    5.0,
    5.1,
])

sigma_plus = np.array([
    23.0,
    3.4,
    8.0,
    4.8,
    4.8,
    9.3,
    2.7,
    2.9,
    3.8,
    5.8,
    5.4,
])

print('Number of lenses =', len(z))
print('Redshift range =', z.min(), 'to', z.max())
print('H0 range =', H0.min(), 'to', H0.max())


In [ ]:
def summarize(name, chain):
    q16, q50, q84 = np.percentile(chain, [16, 50, 84])
    print(f"{name}: {q50:.3f} +{q84 - q50:.3f} -{q50 - q16:.3f}")


def log_prior_linear(theta):
    a, b, lnsig = theta
    if not (40.0 < a < 120.0):
        return -np.inf
    if not (-100.0 < b < 100.0):
        return -np.inf
    if not (-20.0 < lnsig < 5.0):
        return -np.inf
    return 0.0


def log_likelihood_linear(theta):
    a, b, lnsig = theta
    pred = a + b * z
    resid = H0 - pred

    sigma_obs = np.where(resid >= 0.0, sigma_minus, sigma_plus)
    sigma_int = np.exp(lnsig)
    sigma_eff = np.sqrt(sigma_obs**2 + sigma_int**2)

    return np.sum(
        -0.5 * (resid / sigma_eff)**2
        - np.log(sigma_eff)
    )


def log_probability_linear(theta):
    lp = log_prior_linear(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood_linear(theta)


def log_prior_constant(theta):
    c, lnsig = theta
    if not (40.0 < c < 120.0):
        return -np.inf
    if not (-20.0 < lnsig < 5.0):
        return -np.inf
    return 0.0


def log_likelihood_constant(theta):
    c, lnsig = theta
    pred = np.full_like(H0, c)
    resid = H0 - pred

    sigma_obs = np.where(resid >= 0.0, sigma_minus, sigma_plus)
    sigma_int = np.exp(lnsig)
    sigma_eff = np.sqrt(sigma_obs**2 + sigma_int**2)

    return np.sum(
        -0.5 * (resid / sigma_eff)**2
        - np.log(sigma_eff)
    )


def log_probability_constant(theta):
    lp = log_prior_constant(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood_constant(theta)


In [ ]:
# Initial guess from a weighted linear fit
sigma_sym = 0.5 * (sigma_minus + sigma_plus)
w = 1.0 / sigma_sym**2

X = np.column_stack([np.ones_like(z), z])
cov = np.linalg.inv(X.T @ (w[:, None] * X))
beta = cov @ (X.T @ (w * H0))
a0, b0 = beta
c0 = np.average(H0, weights=w)

print('Initial linear guess: a =', a0, 'b =', b0)
print('Initial constant guess: c =', c0)


In [ ]:
# Linear model MCMC
nwalkers = 32
ndim_line = 3
nsteps = 25000

p0_line = np.column_stack([
    a0 + 1e-3 * np.random.randn(nwalkers),
    b0 + 1e-3 * np.random.randn(nwalkers),
    -3.0 + 1e-3 * np.random.randn(nwalkers),
])

sampler_line = emcee.EnsembleSampler(nwalkers, ndim_line, log_probability_linear)
sampler_line.run_mcmc(p0_line, nsteps, progress=False)

flat_line = sampler_line.get_chain(discard=5000, thin=20, flat=True)
logp_line = sampler_line.get_log_prob(discard=5000, thin=20, flat=True)

print('Linear MCMC finished.')

# Constant model MCMC
ndim_const = 2
p0_const = np.column_stack([
    c0 + 1e-3 * np.random.randn(nwalkers),
    -3.0 + 1e-3 * np.random.randn(nwalkers),
])

sampler_const = emcee.EnsembleSampler(nwalkers, ndim_const, log_probability_constant)
sampler_const.run_mcmc(p0_const, nsteps, progress=False)

flat_const = sampler_const.get_chain(discard=5000, thin=20, flat=True)
logp_const = sampler_const.get_log_prob(discard=5000, thin=20, flat=True)

print('Constant MCMC finished.')


In [ ]:
print('LINEAR MODEL')
summarize('a', flat_line[:, 0])
summarize('b', flat_line[:, 1])
summarize('sigma_int', np.exp(flat_line[:, 2]))
print('P(b < 0) =', np.mean(flat_line[:, 1] < 0.0))

print('\nCONSTANT MODEL')
summarize('c', flat_const[:, 0])
summarize('sigma_int', np.exp(flat_const[:, 1]))

# AIC comparison
imax_line = np.argmax(logp_line)
theta_line_best = flat_line[imax_line]
lnL_line = log_likelihood_linear(theta_line_best)
AIC_line = 2 * 3 - 2 * lnL_line

imax_const = np.argmax(logp_const)
theta_const_best = flat_const[imax_const]
lnL_const = log_likelihood_constant(theta_const_best)
AIC_const = 2 * 2 - 2 * lnL_const

delta_AIC = AIC_const - AIC_line

print('\nMODEL COMPARISON')
print('lnL(line)   =', lnL_line)
print('lnL(const)  =', lnL_const)
print('AIC(line)   =', AIC_line)
print('AIC(const)  =', AIC_const)
print('Delta AIC   =', delta_AIC)

if delta_AIC < 2:
    print('Interpretation: models are essentially equivalent')
elif delta_AIC < 6:
    print('Interpretation: some support for the linear model')
elif delta_AIC < 10:
    print('Interpretation: strong support for the linear model')
else:
    print('Interpretation: decisive support for the linear model')


In [ ]:
# Posterior predictive band for the linear fit
zgrid = np.linspace(0.20, 0.78, 500)
idx = np.random.choice(len(flat_line), size=min(10000, len(flat_line)), replace=False)
pred = np.empty((len(idx), len(zgrid)))

for i, j in enumerate(idx):
    pred[i] = flat_line[j, 0] + flat_line[j, 1] * zgrid

median = np.percentile(pred, 50, axis=0)
low68 = np.percentile(pred, 16, axis=0)
high68 = np.percentile(pred, 84, axis=0)
low95 = np.percentile(pred, 2.5, axis=0)
high95 = np.percentile(pred, 97.5, axis=0)

plt.figure(figsize=(8, 6))
plt.fill_between(zgrid, low95, high95, color='lightblue', alpha=0.35, label='95% posterior region')
plt.fill_between(zgrid, low68, high68, color='dodgerblue', alpha=0.45, label='68% posterior region')
plt.plot(zgrid, median, color='navy', lw=2.5, label='Posterior median')
plt.errorbar(z, H0, yerr=[sigma_minus, sigma_plus], fmt='o', color='black', capsize=3, label='Lens measurements')
plt.xlabel(r'$z_d$')
plt.ylabel(r'$H_0\;(\mathrm{km\,s^{-1}\,Mpc^{-1}})$')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Final interpretation
print('This notebook is exploratory only.')
print('The negative slope is suggestive, but the sample is small and the evidence is not decisive.')
print('A robust follow-up would use a larger sample, a hierarchical model, and systematic checks.')
